<a href="https://colab.research.google.com/github/ali-coder7/mathematical-rag/blob/main/Mathematical_RAG_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mathematical RAG: Searching My Maths Notes

I made this notebook to practise a simple idea: upload maths notes as PDFs, ask a question, and find the passages that seem most relevant. The results include the PDF name and page number so I can check them myself.

The project uses Python, a text-embedding model, and NumPy. It is a small first-year learning project. It does not solve or verify mathematics by itself.

## How it works
1. Upload PDF notes.
2. Extract text from each page.
3. Split the text into smaller, overlapping chunks.
4. Convert each chunk and the question into numerical embeddings.
5. Compare the embeddings and rank the closest passages.
6. Check the retrieved passages against the original PDF.


## Step 1: Install the libraries

`pypdf` reads PDF text, `sentence-transformers` creates embeddings, and `reportlab` makes a small sample PDF for testing. No paid API or external text-generation service is used.


In [ ]:
!pip install -q pypdf sentence-transformers reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.7 MB/s eta 0:00:00


## Step 2: Settings
All the settings you might want to change are in this one cell.

In [ ]:
import os
import numpy as np

# Text splitting settings
CHUNK_SIZE = 600       # approximate number of characters in each chunk
CHUNK_OVERLAP = 100    # repeated characters between neighbouring chunks

# Pretrained embedding model
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Number of passages to show for each question
TOP_K = 3

# A rough threshold, not a scientifically calibrated accuracy measure
MIN_SCORE = 0.25

## Step 3: Upload PDF notes

Choose one or more PDFs. If no PDF is uploaded, the notebook creates a short sample document so I can check that the main steps run. The sample is only for testing; it is not a real set of course notes.


In [ ]:
from google.colab import files

uploaded = files.upload()   # opens a file picker; you can select several PDFs
pdf_paths = [name for name in uploaded if name.lower().endswith(".pdf")]

ignored = [name for name in uploaded if name not in pdf_paths]
if ignored:
    print("Ignored (not PDF files):", ignored)
print("PDFs ready:", pdf_paths)

Saving ArtinAlgebra.pdf to ArtinAlgebra.pdf
PDFs ready: ['ArtinAlgebra.pdf']


In [ ]:
# Synthetic notes (plain ASCII, so any PDF library can handle them)
SYNTHETIC_FILENAME = "synthetic_math_notes.pdf"
SYNTHETIC_PAGES = [
    """Quadratic Equations
A quadratic equation has the form a x^2 + b x + c = 0, where a is not zero.
The quadratic formula gives the solutions: x = (-b +/- sqrt(b^2 - 4ac)) / (2a).
The discriminant is D = b^2 - 4ac.
If D > 0 there are two distinct real solutions.
If D = 0 there is exactly one real solution (a repeated root).
If D < 0 there are no real solutions (the solutions are complex numbers).""",
    """Derivatives
The derivative of f at x is the limit of (f(x+h) - f(x)) / h as h approaches 0.
Power rule: if f(x) = x^n, then f'(x) = n x^(n-1).
Product rule: (u v)' = u' v + u v'.
Example: the derivative of x^3 is 3 x^2.""",
    """Pythagorean Theorem
In a right triangle with legs a and b and hypotenuse c, a^2 + b^2 = c^2.
Example: a triangle with legs 3 and 4 has hypotenuse 5, because 9 + 16 = 25.
The converse is also true: if a^2 + b^2 = c^2 holds for the sides of a triangle,
then the triangle has a right angle.""",
]

def make_synthetic_pdf(path, pages):
    # Writes one PDF page per string in `pages`
    from reportlab.pdfgen import canvas
    from reportlab.lib.pagesizes import A4
    pdf = canvas.Canvas(path, pagesize=A4)
    for page_text in pages:
        pdf.setFont("Helvetica", 11)
        y = 800
        for line in page_text.split("\n"):
            pdf.drawString(60, y, line)
            y -= 16
        pdf.showPage()
    pdf.save()

using_synthetic = len(pdf_paths) == 0
if using_synthetic:
    make_synthetic_pdf(SYNTHETIC_FILENAME, SYNTHETIC_PAGES)
    pdf_paths = [SYNTHETIC_FILENAME]
    print("No PDF uploaded -> using the SYNTHETIC example:", SYNTHETIC_FILENAME)
    print("(Made-up notes for testing only; not a real document.)")

## Step 4: Extract text page by page

The notebook keeps the file name and page number with each page of text. This helps me find the original passage later. Some mathematical symbols may not extract correctly, and scanned PDFs need OCR, which is not included here.


In [ ]:
from pypdf import PdfReader

def extract_pages(pdf_path):
    # Returns a list of {"filename", "page", "text"}, one item per page that contains text
    reader = PdfReader(pdf_path)
    filename = os.path.basename(pdf_path)
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):   # page numbers start at 1
        text = (page.extract_text() or "").strip()
        if text:
            pages.append({"filename": filename, "page": page_number, "text": text})
    return pages

In [ ]:
all_pages = []
for path in pdf_paths:
    try:
        pages = extract_pages(path)
    except Exception as error:
        print(f"Could not read {path}: {error}")
        continue
    print(f"{os.path.basename(path)}: {len(pages)} page(s) with text")
    all_pages.extend(pages)

if not all_pages:
    raise RuntimeError("No text could be extracted. Scanned PDFs have no text layer; try another PDF.")

# Look at the first page to check the extraction quality (especially the formulas!)
print("\n--- First 400 characters of the first page ---")
print(all_pages[0]["text"][:400])

ArtinAlgebra.pdf: 552 page(s) with text

--- First 400 characters of the first page ---
Contents
P re fa c e  xi
1 Matrices
1.1 The Basic O p e ra tio n s........................................................................................... 1
1.2 Row Reduction........................................................................................... .. 10
1.3 The Matrix T ranspose........................................................................................... 17
1.4 D


## Step 5: Split each page into chunks

I use chunks of 600 characters with a 100-character overlap. The overlap helps keep some context when a sentence falls near a chunk boundary. Chunks stay within their original page, so each result can still point to a page. This simple approach can split a formula or sentence in an awkward place.


In [ ]:
def split_into_chunks(text, chunk_size, overlap):
    # Cuts one string into overlapping pieces
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    step = chunk_size - overlap
    pieces = []
    for start in range(0, len(text), step):
        piece = text[start:start + chunk_size].strip()
        if piece:
            pieces.append(piece)
        if start + chunk_size >= len(text):   # this piece already reaches the end
            break
    return pieces

def chunk_pages(pages, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    # Chunks every page and remembers where each chunk came from
    chunks = []
    for page in pages:
        for piece in split_into_chunks(page["text"], chunk_size, overlap):
            chunks.append({"filename": page["filename"], "page": page["page"], "text": piece})
    return chunks

In [ ]:
chunks = chunk_pages(all_pages)
print(f"{len(all_pages)} page(s) -> {len(chunks)} chunk(s)")
print("\nExample chunk:", chunks[0]["filename"], "| page", chunks[0]["page"])
print(chunks[0]["text"])

552 page(s) -> 3237 chunk(s)

Example chunk: ArtinAlgebra.pdf | page 1
Contents
P re fa c e  xi
1 Matrices
1.1 The Basic O p e ra tio n s........................................................................................... 1
1.2 Row Reduction........................................................................................... .. 10
1.3 The Matrix T ranspose........................................................................................... 17
1.4 D eterm inants.........................................................................................................  18
1.5 P erm utations...........................................................


## Step 6: Create embeddings

An embedding is a list of numbers that represents text in a form a computer can compare. The model used here returns 384 numbers for each text. It is a general-purpose English model, not a model designed specifically for mathematics, so similar wording does not guarantee that two passages mean the same thing mathematically.


In [ ]:
print("chunks:", len(chunks))
print("chunk_embeddings exists:", "chunk_embeddings" in globals())
print("embedding variables:", [
    name for name in globals()
    if "embed" in name.lower()
])

chunks: 3237
chunk_embeddings exists: False
embedding variables: ['EMBEDDING_MODEL', 'embed_texts']


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDING_MODEL)   # downloads the model the first time

def embed_texts(texts):
    # Returns a NumPy array with one row (vector of length 1) per text
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Step 7: Search for relevant passages

The question is converted to an embedding and compared with the chunk embeddings using cosine similarity. Because the vectors are normalised, their dot product gives the similarity score used for ranking. The notebook shows the highest-ranked passages and their sources.

NumPy is enough for this small project. A specialised vector-search library would be more useful for a much larger collection.


In [ ]:
chunk_embeddings = embed_texts(
    [chunk["text"] for chunk in chunks]
)

print("Shape:", chunk_embeddings.shape)

Shape: (3237, 384)


In [ ]:
def retrieve(question, chunks, chunk_embeddings, top_k=TOP_K):
    # Returns the top_k best chunks as a list of dicts, best first
    question_vector = embed_texts([question])[0]
    scores = chunk_embeddings @ question_vector          # one score per chunk
    best_indices = np.argsort(-scores)[:top_k]           # indices of the highest scores
    results = []
    for rank, index in enumerate(best_indices, start=1):
        results.append({
            "rank": rank,
            "score": float(scores[index]),
            "filename": chunks[index]["filename"],
            "page": chunks[index]["page"],
            "text": chunks[index]["text"],
        })
    return results

def show_passages(results):
    for result in results:
        print(f"[{result['rank']}] {result['filename']} | page {result['page']} | score {result['score']:.3f}")
        print(result["text"])
        print("-" * 60)

In [ ]:
sample_question = "What is row reduction in matrices?"
show_passages(retrieve(sample_question, chunks, chunk_embeddings))

[1] ArtinAlgebra.pdf | page 25 | score 0.653
en in the form
X3 =  c ,  x i  =  3 +  c ,  X2 =  1  —  3c, X4 =  1,
where c is arbitrary.
We now go back to row reduction of an arbitrary matrix. It is not hard to see that, by
a sequence of row operations, any matrix M can be reduced to what is called a row echelon
matrix. The end result of our reduction of (1.2.8) is an example. Here is the definition: A 
row echelon matrix  is a matrix that has these properties:
(1 .2 .1 2)
(a) If ( row i) of M is zero, then (row j) is zero for all j  > i.
(b) If (ro w i) isn’t zero, its first nonzero entry is 1. This entry is called a pivot.
(c) If (row (
------------------------------------------------------------
[2] ArtinAlgebra.pdf | page 24 | score 0.648
12 Chapter 1  Matrices
We now perform elementary row operations (1.2.5) on a matrix M, with the aim of 
ending up with a simpler matrix:
M sequence of ope rations
Since each elementary operation is obtained by multiplying by an elementary matrix, w

## Step 8: Read the retrieved passages

This version does not use a text-generation API. It shows the passages found by the search, their similarity scores, and the PDF page where each passage came from. I then compare the result with the original notes before trusting it.


In [ ]:
NOT_ENOUGH_INFO = "The notes may not contain enough information to answer this question reliably."

## Step 9: Ask questions

The PDFs are processed once. After that, I can ask several questions without rebuilding the search index each time.


In [ ]:
def ask(question, top_k=TOP_K):
    if not question.strip():
        print("Please type a question.")
        return

    results = retrieve(question, chunks, chunk_embeddings, top_k)
    print("QUESTION:", question, "\n")
    show_passages(results)

    if not results or results[0]["score"] < MIN_SCORE:
        print("NOTE:", NOT_ENOUGH_INFO)
    else:
        print("NEXT STEP: Check the passages and formulas against the original PDF.")

In [ ]:
# Type questions one after another. Press Enter on an empty line to stop.
while True:
    user_question = input("Your question (empty line to stop): ").strip()
    if not user_question:
        break
    ask(user_question)
    print("=" * 60)

Your question (empty line to stop): What is an equivalence relation?
QUESTION: What is an equivalence relation? 

[1] ArtinAlgebra.pdf | page 65 | score 0.556
y be presented with just one of the two.
Proposition 2.7.4 An equivalence relation on a set S determines a partition of S, and
Proof  Given a partition of S, the corresponding equivalence relation is defined by the rule 
that a ~  b if a and b lie in the same subset of the partition. The axioms for an equivalence 
relation are obviously satisfied. Conversely, given an equivalence relation, one defines a 
partition this way: The subset that contains a is the set of all elements b such that a ~  b. This 
subset is called the equivalence class  of a. We’ll denote it by Ca here:
(2.7.2)
conversely
------------------------------------------------------------
[2] ArtinAlgebra.pdf | page 65 | score 0.530
Odd  partition the set of integers. With the usual notation, 
the sets
form a partition of the symmetric group S3.
• An equivalence re


KeyboardInterrupt



## Step 10: Run basic checks

These checks test the mechanics of the notebook using a small, made-up three-page maths PDF. They check text extraction, chunking, embeddings, retrieval, and source references. Passing these checks does not prove that the search will always find the right answer in real course notes.


In [ ]:
# Tests 1: extraction and chunking
test_results = []

def check(description, condition):
    passed = bool(condition)
    test_results.append(passed)
    print(("PASS  " if passed else "FAIL  ") + description)

TEST_PDF = "test_synthetic_notes.pdf"
make_synthetic_pdf(TEST_PDF, SYNTHETIC_PAGES)
test_pages = extract_pages(TEST_PDF)

print("-- Extraction --")
check("3 pages with text were found", len(test_pages) == 3)
check("page numbers are 1, 2, 3", [p["page"] for p in test_pages] == [1, 2, 3])
check("file name is kept on every page", all(p["filename"] == TEST_PDF for p in test_pages))
check("page 1 text contains 'discriminant'", "discriminant" in test_pages[0]["text"].lower())
check("page 3 text contains 'hypotenuse'", "hypotenuse" in test_pages[2]["text"].lower())

print("\n-- Chunking --")
long_text = "abcdefghij" * 100                       # 1000 characters, no spaces
test_chunks = split_into_chunks(long_text, chunk_size=300, overlap=50)
check("no chunk is longer than chunk_size", all(len(c) <= 300 for c in test_chunks))
check("first chunk starts at the beginning of the text", long_text.startswith(test_chunks[0]))
check("last chunk reaches the end of the text", long_text.endswith(test_chunks[-1]))
check("neighbouring chunks share exactly 50 characters",
      all(test_chunks[i][-50:] == test_chunks[i + 1][:50] for i in range(len(test_chunks) - 1)))

try:
    split_into_chunks("abc", chunk_size=10, overlap=10)
    check("overlap >= chunk_size is rejected", False)
except ValueError:
    check("overlap >= chunk_size is rejected", True)

small_chunks = chunk_pages(test_pages, chunk_size=300, overlap=50)
check("chunks keep their file name and page number",
      all(c["filename"] == TEST_PDF and c["page"] in (1, 2, 3) for c in small_chunks))

-- Extraction --
PASS  3 pages with text were found
PASS  page numbers are 1, 2, 3
PASS  file name is kept on every page
PASS  page 1 text contains 'discriminant'
PASS  page 3 text contains 'hypotenuse'

-- Chunking --
PASS  no chunk is longer than chunk_size
PASS  first chunk starts at the beginning of the text
PASS  last chunk reaches the end of the text
PASS  neighbouring chunks share exactly 50 characters
PASS  overlap >= chunk_size is rejected
PASS  chunks keep their file name and page number


In [ ]:
# Tests 2: embeddings
print("-- Embeddings --")
sample_texts = [
    "The quadratic formula solves a x^2 + b x + c = 0.",
    "How do I solve a quadratic equation?",
    "A recipe for chocolate cake.",
]
sample_vectors = embed_texts(sample_texts)
check("one vector per text", sample_vectors.shape[0] == len(sample_texts))
check("vector length matches the model", sample_vectors.shape[1] == model.get_sentence_embedding_dimension())
check("vectors are normalised (length 1)", np.allclose(np.linalg.norm(sample_vectors, axis=1), 1.0, atol=1e-3))

similarity_related = float(sample_vectors[0] @ sample_vectors[1])
similarity_unrelated = float(sample_vectors[0] @ sample_vectors[2])
check(f"related texts score higher than unrelated ones ({similarity_related:.2f} vs {similarity_unrelated:.2f})",
      similarity_related > similarity_unrelated)

-- Embeddings --
PASS  one vector per text
PASS  vector length matches the model
PASS  vectors are normalised (length 1)
PASS  related texts score higher than unrelated ones (0.63 vs 0.05)


/tmp/ipykernel_6292/2371969027.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  check("vector length matches the model", sample_vectors.shape[1] == model.get_sentence_embedding_dimension())


In [ ]:
# Tests 3: retrieval and source attribution
test_embeddings = embed_texts([c["text"] for c in small_chunks])

print("-- Retrieval --")
expected_pages = {
    "What is the discriminant of a quadratic equation?": 1,
    "State the power rule for derivatives.": 2,
    "What is the hypotenuse of a right triangle with legs 3 and 4?": 3,
}
retrieval_hits = 0
retrieval_hits = 0
for test_question, expected_page in expected_pages.items():
    top = retrieve(test_question, small_chunks, test_embeddings, top_k=3)
    hit = bool(top) and top[0]["page"] == expected_page
    retrieval_hits += int(hit)
    check(f"'{test_question}' -> best result is page {expected_page} (got page {top[0]['page']})",
          hit)

print("\n-- Source attribution --")
page_text_lookup = {(p["filename"], p["page"]): p["text"] for p in test_pages}
top = retrieve("What is the power rule?", small_chunks, test_embeddings, top_k=3)
check("every result has filename, page, text and score",
      all(all(key in r for key in ("filename", "page", "text", "score")) for r in top))
check("every filename/page pair exists in the PDF",
      all((r["filename"], r["page"]) in page_text_lookup for r in top))
check("every passage really appears on the page it cites",
      all(r["text"] in page_text_lookup[(r["filename"], r["page"])] for r in top))
check("results are sorted from highest to lowest score",
      [r["score"] for r in top] == sorted([r["score"] for r in top], reverse=True))

print(f"\n{sum(test_results)} of {len(test_results)} checks passed.")

-- Retrieval --
PASS  'What is the discriminant of a quadratic equation?' -> best result is page 1 (got page 1)
PASS  'State the power rule for derivatives.' -> best result is page 2 (got page 2)
PASS  'What is the hypotenuse of a right triangle with legs 3 and 4?' -> best result is page 3 (got page 3)

-- Source attribution --
PASS  every result has filename, page, text and score
PASS  every filename/page pair exists in the PDF
PASS  every passage really appears on the page it cites
PASS  results are sorted from highest to lowest score

22 of 22 checks passed.


## Step 11: Try a few questions

If I uploaded my own PDF, I can add questions whose answers I know and compare the retrieved pages with the document. If I used the sample PDF, the first three questions are covered by the sample notes and the last question is not. This is a demonstration, not a formal accuracy study.


In [ ]:
my_questions = ["What is an equivalence class?",
    "How does an equivalence relation determine a partition?",
    "What is row reduction in matrices?",]   # for your own PDF, e.g. ["State the definition of a group.", "What does Theorem 2 say?"]

if using_synthetic:
    print("SYNTHETIC EXAMPLE (made-up notes, not a real document)\n")
    demo_questions = [
        "What does the discriminant tell us about the solutions of a quadratic equation?",
        "What is the power rule for derivatives?",
        "How long is the hypotenuse of a right triangle with legs 3 and 4?",
        "What is the Fourier transform of a Gaussian function?",   # not in the notes
    ]
else:
    demo_questions = my_questions
    if not demo_questions:
        print("Add some questions to my_questions above and run this cell again.")

for demo_question in demo_questions:
    ask(demo_question)
    print("=" * 60)

QUESTION: What is an equivalence class? 

[1] ArtinAlgebra.pdf | page 65 | score 0.593
y be presented with just one of the two.
Proposition 2.7.4 An equivalence relation on a set S determines a partition of S, and
Proof  Given a partition of S, the corresponding equivalence relation is defined by the rule 
that a ~  b if a and b lie in the same subset of the partition. The axioms for an equivalence 
relation are obviously satisfied. Conversely, given an equivalence relation, one defines a 
partition this way: The subset that contains a is the set of all elements b such that a ~  b. This 
subset is called the equivalence class  of a. We’ll denote it by Ca here:
(2.7.2)
conversely
------------------------------------------------------------
[2] ArtinAlgebra.pdf | page 66 | score 0.523
re generally. When we want to re gard a subset U of S as 
an element of a set of subsets of S, we denote it by [U].
When an equivalence relation on S is given, the equivalence classes form a partition, 
and

## Final results and what I learned

The extraction and chunking unit tests passed on the synthetic test document. The retrieval pipeline was also tested with questions about a real mathematics PDF, ArtinAlgebra.pdf.

For the questions tested, the system retrieved relevant passages and preserved their source filenames and page numbers.

The implementation uses sentence embeddings and cosine similarity to rank passages by relevance.

Through this project, I learned how to extract text from PDFs, split documents into overlapping chunks, represent text as numerical vectors, and retrieve relevant passages based on semantic similarity.

 I also learned the importance of testing individual components and checking retrieved results against the original source.

These results demonstrate that the pipeline works for the tested cases; they do not establish a general accuracy score.

 The current implementation focuses on semantic retrieval and does not generate answers using a large language model.